> ### Load & Explore


##### Init


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType, StructType, StructField


##### Reading from CSV File

In [0]:
file_path = "/Volumes/db_academy_project/default/source/lego_sets.csv"


df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(file_path)
    
)


##### Display the first 10 rows

In [0]:
display(df.limit(10))

##### Schema , column names and data types

In [0]:
df.printSchema()

##### Total number of rows

In [0]:
total_rows = df.count()
print(f"Total number of rows : {total_rows}")

##### Missing values per column

In [0]:

missing_values_count = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

missing_values_count_pd = missing_values_count.toPandas().iloc[0].sort_values(ascending=False)


display(missing_values_count_pd)

> ### Transformations

##### Handling missing values

I handled missing values selectively because each null has a different meaning. For `themeGroup` I filled with `"Unknown"` since there are only 2 missing values and it's a categorical column. For `agerange_min` I filled with the median (6) instead of dropping rows because dropping would have removed 63% of the data which is too much. I chose the median over the mean because some sets have very high age ratings (16-18), which would pull the mean up and make it less accurate.

In [0]:


# Before
print("data before filling:")
print(f"  themeGroup nulls:  {df.filter(F.col('themeGroup').isNull()).count()}")
print(f"  agerange_min nulls: {df.filter(F.col('agerange_min').isNull()).count()}")

# Calculate median
median_age = df.approxQuantile("agerange_min", [0.5], 0.01)[0]

# Fill missing values
df_cleaned = df.fillna({
    "themeGroup": "Unknown",
    "agerange_min": int(median_age)
})

print() # blank line 

# After
print("data after filling:")
print(f"  themeGroup nulls:  {df_cleaned.filter(F.col('themeGroup').isNull()).count()}")
print(f"  agerange_min nulls: {df_cleaned.filter(F.col('agerange_min').isNull()).count()}")


##### Adding a new column

In [0]:
df = df.withColumn("age_range",
                   F.when(F.col("agerange_min") > 18, "Over 18")
                   .when(F.col("agerange_min") > 10, "10 to 17")
                   .when(F.col("agerange_min") > 5, "5 to 9")
                   .otherwise("1 to 4")
                   )

display(df.limit(5))

##### Filtering the Data

In [0]:
df = df.filter((F.col("year") > 2000) & (F.col("pieces").isNotNull()))


display(df.limit(5))